# Modelling: predicting the age of an abalone

In this notebook, we train models to predict the number of `Rings` of an abalone
(age in years ≈ `Rings` + 1.5) and track the experiments with MLflow.

MLflow setup:
* Tracking server: no
* Backend store and artifacts store: local (`mlflow.db` and `mlruns/` in the `notebooks` folder, not pushed to the repository)

The experiments can be compared in the MLflow UI, by running from the `notebooks` folder:

```bash
uv run mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5002
```

In [1]:
from pathlib import Path

import kagglehub
import mlflow
import numpy as np
import pandas as pd
import scipy.sparse
from mlflow import MlflowClient
from sklearn.base import RegressorMixin
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

TARGET = "Rings"
CATEGORICAL_COLS = ["Sex"]
NUMERICAL_COLS = [
    "Length",
    "Diameter",
    "Height",
    "Whole weight",
    "Shucked weight",
    "Viscera weight",
    "Shell weight",
]
TEST_SIZE = 0.2
RANDOM_STATE = 42

# 1 - Load data

In [2]:
def load_data(path: Path) -> pd.DataFrame:
    """Load the Abalone dataset from a CSV file."""
    return pd.read_csv(path)


dataset_path = kagglehub.dataset_download("rodolfomendes/abalone-dataset")
data_path = Path(dataset_path) / "abalone.csv"

df = load_data(data_path)
df.head()

,Sex,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


# 2 - Prepare the data

## 2-1 Filter outliers

As seen in the EDA, a few abalones have an impossible height (0 or above 0.4). We remove them.

In [3]:
MIN_HEIGHT = 0
MAX_HEIGHT = 0.4


def filter_outliers(
    df: pd.DataFrame, min_height: float = MIN_HEIGHT, max_height: float = MAX_HEIGHT
) -> pd.DataFrame:
    """Remove the abalones with a height of 0 or above the maximal height."""
    return df[(df["Height"] > min_height) & (df["Height"] <= max_height)].reset_index(
        drop=True
    )


df = filter_outliers(df)
df.shape

(4173, 9)

## 2-2 Split the data

We keep 20% of the abalones to evaluate the models on data they have not seen during training.

In [4]:
train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=RANDOM_STATE)
print(f"Train: {train_df.shape}, test: {test_df.shape}")

Train: (3338, 9), test: (835, 9)


## 2-3 Prepare features

`Sex` is a categorical feature: the `DictVectorizer` one-hot encodes it, and keeps the numerical features as they are.
The `DictVectorizer` is fitted on the training data only, and then reused to transform the test data.

In [5]:
def extract_x_y(
    df: pd.DataFrame,
    categorical_cols: list[str] | None = None,
    numerical_cols: list[str] | None = None,
    dv: DictVectorizer | None = None,
    with_target: bool = True,
) -> tuple[scipy.sparse.csr_matrix, np.ndarray | None, DictVectorizer]:
    """Extract X (and y) from the dataframe, fitting a DictVectorizer if none is given."""
    if categorical_cols is None:
        categorical_cols = CATEGORICAL_COLS
    if numerical_cols is None:
        numerical_cols = NUMERICAL_COLS
    dicts = df[categorical_cols + numerical_cols].to_dict(orient="records")

    if dv is None:
        dv = DictVectorizer()
        dv.fit(dicts)
    x = dv.transform(dicts)

    y = df[TARGET].values if with_target else None
    return x, y, dv


X_train, y_train, dv = extract_x_y(train_df)
X_test, y_test, _ = extract_x_y(test_df, dv=dv)
dv.feature_names_

['Diameter',
 'Height',
 'Length',
 'Sex=F',
 'Sex=I',
 'Sex=M',
 'Shell weight',
 'Shucked weight',
 'Viscera weight',
 'Whole weight']

# 3 - Train models

We train three models:
- a linear regression, to have a baseline performance;
- a random forest and an XGBoost (gradient boosting), which can capture the non-linear relations between the measurements and the number of rings observed in the EDA.

In [6]:
def train_model(
    x_train: scipy.sparse.csr_matrix, y_train: np.ndarray, model: RegressorMixin
) -> RegressorMixin:
    """Train a regression model."""
    model.fit(x_train, y_train)
    return model


models = {
    "LinearRegression": LinearRegression(),
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=200, min_samples_leaf=5, random_state=RANDOM_STATE
    ),
    "XGBRegressor": XGBRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=4, random_state=RANDOM_STATE
    ),
}

for model_name, model in models.items():
    models[model_name] = train_model(X_train, y_train, model)

# 4 - Evaluate models

We evaluate the models with the RMSE (in number of rings), on train and test data.

In [7]:
def predict_rings(
    input_data: scipy.sparse.csr_matrix, model: RegressorMixin
) -> np.ndarray:
    """Predict the number of rings of the abalones."""
    return model.predict(input_data)


def evaluate_model(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Compute the RMSE of the predictions."""
    return root_mean_squared_error(y_true, y_pred)

In [8]:
pd.DataFrame(
    {
        model_name: {
            "train_rmse": evaluate_model(y_train, predict_rings(X_train, model)),
            "test_rmse": evaluate_model(y_test, predict_rings(X_test, model)),
        }
        for model_name, model in models.items()
    }
).T.round(3)

,train_rmse,test_rmse
LinearRegression,2.167,2.229
RandomForestRegressor,1.488,2.199
XGBRegressor,1.614,2.207


# 5 - Log model parameters to MLflow

Now that all our functions are built and tested, we run the whole training pipeline in one MLflow run per model,
logging its parameters, metrics and model, and we register each model in the MLflow Model Registry.

In [9]:
EXPERIMENT_NAME = "abalone-age"
REGISTERED_MODEL_NAME = "abalone_age_model"
# Types of the tree-based models to trust when saving them with MLflow (the models are trained here)
TRUSTED_TYPES = [
    "sklearn.tree._tree.Tree",
    "xgboost.core.Booster",
    "xgboost.sklearn.XGBRegressor",
]

mlflow.set_experiment(EXPERIMENT_NAME)


def train_and_log_model(model: RegressorMixin) -> dict:
    """Run the training pipeline of a model in an MLflow run, and register the model in the Model Registry."""
    with mlflow.start_run():
        # Set tags for the run
        mlflow.set_tag("model", type(model).__name__)
        mlflow.set_tag("data", "rodolfomendes/abalone-dataset")

        # Load and filter data
        df = filter_outliers(load_data(data_path))
        train_df, test_df = train_test_split(
            df, test_size=TEST_SIZE, random_state=RANDOM_STATE
        )
        mlflow.log_params(
            {
                "test_size": TEST_SIZE,
                "split_random_state": RANDOM_STATE,
                "max_height": MAX_HEIGHT,
            }
        )

        # Extract X and y
        X_train, y_train, dv = extract_x_y(train_df)
        X_test, y_test, _ = extract_x_y(test_df, dv=dv)

        # Train model
        model = train_model(X_train, y_train, model)
        mlflow.log_params(model.get_params())

        # Evaluate model on train and test sets
        train_rmse = evaluate_model(y_train, predict_rings(X_train, model))
        test_rmse = evaluate_model(y_test, predict_rings(X_test, model))
        mlflow.log_metrics({"train_rmse": train_rmse, "test_rmse": test_rmse})

        # Log the model
        model_info = mlflow.sklearn.log_model(
            model, name="model", skops_trusted_types=TRUSTED_TYPES
        )

        # Register the model in the MLflow Model Registry
        result = mlflow.register_model(model_info.model_uri, REGISTERED_MODEL_NAME)

    print(
        f"{type(model).__name__} | train RMSE: {train_rmse:.2f} | test RMSE: {test_rmse:.2f} | version: {result.version}"
    )
    return {"version": result.version, "test_rmse": test_rmse}


runs = {model_name: train_and_log_model(model) for model_name, model in models.items()}

LinearRegression | train RMSE: 2.17 | test RMSE: 2.23 | version: 1


RandomForestRegressor | train RMSE: 1.49 | test RMSE: 2.20 | version: 2


XGBRegressor | train RMSE: 1.61 | test RMSE: 2.21 | version: 3


We mark the model with the lowest test RMSE as the production one with the `production` alias.
This will help us retrieve it for predictions.

In [10]:
best_model_name = min(runs, key=lambda model_name: runs[model_name]["test_rmse"])

client = MlflowClient()
client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME,
    alias="production",
    version=runs[best_model_name]["version"],
)
print(
    f"Production model: {best_model_name} (version {runs[best_model_name]['version']})"
)

Production model: RandomForestRegressor (version 2)


# 6 - Predict

We can now load the production model from the registry and predict the age of new abalones,
using only their physical measurements.

In [11]:
# New abalones, without their number of rings
new_abalones = test_df.drop(columns=TARGET).head()
X_new, _, _ = extract_x_y(new_abalones, dv=dv, with_target=False)

# Load production model
model = mlflow.sklearn.load_model(f"models:/{REGISTERED_MODEL_NAME}@production")

# Make predictions
predicted_rings = predict_rings(X_new, model)
new_abalones.assign(
    predicted_rings=predicted_rings.round(1),
    predicted_age=(predicted_rings + 1.5).round(1),
)

,Sex,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,predicted_rings,predicted_age
501,F,0.620,0.520,0.225,1.1835,0.3780,0.2700,0.3950,15.4,16.9
463,I,0.220,0.165,0.055,0.0545,0.0215,0.0120,0.0200,5.1,6.6
1424,F,0.745,0.570,0.215,2.2500,1.1565,0.4460,0.5580,11.2,12.7
1461,M,0.505,0.385,0.105,0.5525,0.2390,0.1245,0.1555,8.6,10.1
3023,I,0.495,0.375,0.115,0.5070,0.2410,0.1030,0.1500,8.2,9.7


# Conclusions

| Model | Train RMSE | Test RMSE |
|---|---|---|
| Linear regression | 2.17 | 2.23 |
| Random forest | 1.49 | 2.20 |
| XGBoost | 1.61 | 2.21 |

- The linear regression already gives a good baseline, with an error of about 2.2 rings on the test set.
- The random forest and XGBoost are only slightly better on the test set (≈ 0.03 ring), while their train error is much lower: they overfit the training data.
- The random forest has the lowest test RMSE, so it is registered as the `production` model in the MLflow Model Registry.
- All the runs (parameters, metrics and models) are tracked in MLflow and can be compared in the MLflow UI.